# Phase 9 Inventory Decision

Convert Phase 7 point forecasts into reorder decisions and quantify policy cost.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from retail_demand.inventory.backtest import backtest_inventory, compute_forecast_error_std
from retail_demand.inventory.policies import (
    policy_ml_95_service,
    policy_ml_99_service,
    policy_naive_rolling_mean,
    policy_seasonal_naive,
)

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "file:///tmp/retail-demand-mlruns")
BEST_RUN_ID = os.getenv("BEST_RUN_ID", "28cb96fe2b5542c797b048dffbb6eae6")
TABLE_DIR = Path(os.getenv("INVENTORY_TABLE_DIR", "reports/tables"))
FIGURE_DIR = Path(os.getenv("INVENTORY_FIGURE_DIR", "reports/figures"))
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Inventory analysis run: {BEST_RUN_ID}")

In [ ]:
predictions_csv = os.getenv("INVENTORY_PREDICTIONS_CSV")
if predictions_csv:
    preds = pd.read_csv(predictions_csv, parse_dates=["date"])
else:
    import mlflow

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    artifact = mlflow.artifacts.download_artifacts(
        run_id=BEST_RUN_ID, artifact_path="forecast_vs_actual.csv"
    )
    preds = pd.read_csv(artifact, parse_dates=["date"])

sales_history = pd.read_csv(os.environ["INVENTORY_SALES_CSV"], parse_dates=["date"])
prices = pd.read_csv(os.environ["INVENTORY_PRICES_CSV"], parse_dates=["week_start"])
products = pd.read_csv(os.environ["INVENTORY_PRODUCTS_CSV"])
print({"predictions": len(preds), "sales_history": len(sales_history)})

In [ ]:
forecast_error_std = compute_forecast_error_std(preds)
forecast_error_std.head()

In [ ]:
LEAD_TIME_DAYS = int(os.getenv("INVENTORY_LEAD_TIME_DAYS", "7"))
policies = {
    "baseline_rolling_mean": policy_naive_rolling_mean(window=7),
    "baseline_seasonal_naive": policy_seasonal_naive(),
    "ML_95": policy_ml_95_service(lead_time=LEAD_TIME_DAYS),
    "ML_99": policy_ml_99_service(lead_time=LEAD_TIME_DAYS),
}
params = {"lead_time_days": LEAD_TIME_DAYS, "products_df": products}
list(policies)

In [ ]:
backtest = backtest_inventory(preds, sales_history, prices, policies, params)
per_sku = backtest["per_sku"]
summary = backtest["summary"]
per_sku.to_csv(TABLE_DIR / "inventory_per_sku.csv", index=False)
summary.to_csv(TABLE_DIR / "inventory_cost_comparison.csv", index=False)
summary

In [ ]:
plot_summary = summary.sort_values("total_cost")
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(plot_summary["policy"], plot_summary["total_cost"])
ax.set_ylabel("Total cost, USD")
ax.set_title("Inventory Policy Cost Comparison")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "inventory_cost_comparison.png", dpi=150)
plt.close(fig)
plot_summary

In [ ]:
service_grid = {"85%": 1.036, "90%": 1.282, "95%": 1.645, "97.5%": 1.960, "99%": 2.326}
sensitivity_rows = []
for label, z_score in service_grid.items():
    result = backtest_inventory(
        preds,
        sales_history,
        prices,
        {f"ML_{label}": policy_ml_95_service(lead_time=LEAD_TIME_DAYS, z=z_score)},
        params,
    )["summary"].iloc[0]
    sensitivity_rows.append({"service_level": label, "total_cost": result["total_cost"]})
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(TABLE_DIR / "inventory_service_sensitivity.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sensitivity["service_level"], sensitivity["total_cost"], marker="o")
ax.set_xlabel("Target service level")
ax.set_ylabel("Total cost, USD")
ax.set_title("Inventory Cost Sensitivity")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "inventory_service_sensitivity.png", dpi=150)
plt.close(fig)
sensitivity

In [ ]:
baseline = per_sku[per_sku["policy"] == "baseline_rolling_mean"]
ml95 = per_sku[per_sku["policy"] == "ML_95"]
category_delta = baseline.merge(
    ml95,
    on=["store_id", "sku_id", "category"],
    suffixes=("_baseline", "_ml95"),
)
category_delta["cost_delta_ml95_vs_baseline"] = (
    category_delta["total_cost_ml95"] - category_delta["total_cost_baseline"]
)
category_delta = (
    category_delta.groupby("category", dropna=False, observed=True)
    .agg(cost_delta_ml95_vs_baseline=("cost_delta_ml95_vs_baseline", "sum"))
    .reset_index()
    .sort_values("cost_delta_ml95_vs_baseline")
)
category_delta.to_csv(TABLE_DIR / "inventory_category_delta.csv", index=False)
category_delta

## Interpretation Summary

Fill in after execution: lowest-cost policy, ML_95 cost delta versus baseline_rolling_mean, ML_99 incremental service-level gain, and categories where ML-driven replenishment saves or spends the most. Treat the result as a decision simulation, not a production ordering recommendation.